# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: Pre Tournament

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick
2. Advance simulation based optimisation process 
2. Nice to have: bench tricks

# 0. Prerequistes

In [175]:
pip install mip==1.16rc0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [176]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm
import joblib

os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/pre_tourny/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/'
mock_data_dir = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/mock_data/'

In [177]:
# Optimisation Strategy
opt_strat = 'efp' # Options: 'efp_old' (previous player expected fantasy points), 'efp' (new player expected fantasy points appraoch), 'range' (new player fantasy point distribution approach)
current_rnd = 1 # Current Round of the Tournament

# 1. EFP Old Approach

# a. Data Extraction 

In [178]:
# Previous Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp_old':
    # Data Extraction 
    # Current Round & Optimal Number of Rounds
    current_round = current_rnd
    opt_round = 3

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
    team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 10.51, player_df_raw["exp_pts"] + 4.53)
    player_df_raw["weight"] = 1

    # Aggregate by player name (calculate total expected points for each player for x number of rounds)
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    exp_points=('exp_points',"sum"))

    # Create dataframe for next round expected points (For captain selection)
    player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    next_rnd_exp_points=('exp_points',"sum"))
    player_df_next_rnd = player_df_next_rnd[["Name", "Team", "next_rnd_exp_points"]]

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


# b. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [179]:
# EFP Optimisation Variable Setup
if opt_strat == 'efp_old':
    points = player_df["exp_points"]
    price = player_df["Price"]

    weight = player_df["weight"]
    available = player_df["Available"]
    in_team = player_df["In_Team"]
    wk_weight = player_df["Wk_f"]
    bat_weight = player_df["Bat_f"]
    bowl_weight = player_df["Bowl_f"]

    play_cnt, total_player = 12, range(len(price))
    # team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
    wk_cnt, total_wk = 1, range(len(price))
    bat_cnt, total_bat = 6, range(len(price))
    bowl_cnt, total_bowl = 5, range(len(price))
    budget, total_budget = 1783500, range(len(price))

    # MIP Optimsation Setup
    # a. initialize optimisation parameters
    m = Model("knapsack")
    # x is player selected variable, y is captain selected variable (2x points)
    x = [m.add_var(var_type=BINARY) for i in total_player]
    y = [m.add_var(var_type=BINARY) for i in total_player]

    # b. define objective function
    m.objective = maximize(xsum(points[i]*x[i] + points[i]*y[i] for i in total_player))

    # c. define constraints
        # Player selection constraints
    m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
    m += xsum(y[i] for i in total_player) == 1  # Only one captain

    # captain can only be chosen among selected players
    for i in total_player:
        m += y[i] <= x[i]

        # Team composition constraints  
    m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
    m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
    m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
    m += xsum(price[i] * x[i] for i in total_budget) <= budget
    m += xsum(available[i] * x[i] for i in total_player) == play_cnt
    # m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

    # d. solve optimisation
    m.optimize() 
    selected = [i for i in total_player if x[i].x >= 0.99]
    captained = [i for i in total_player if y[i].x >= 0.99]
    
    # Optimisation Results
    print("selected items: {}".format(selected))
    print("Captain selected: {}".format(captained))

    # Optimal Team Output
    sel_player_df = player_df.iloc[selected]
    sel_captain_df = player_df.iloc[captained]
    sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
    print("Total Expected Points:", sum(sel_player_df["exp_points"]))
    print("Total Next Round Points:", sum(sel_player_df["next_rnd_exp_points"]))
    print("Total Team Cost:", sum(sel_player_df["Price"]))
    print("Number of Wk:", sum(sel_player_df["Wk_f"]))
    print("Number of Bat:", sum(sel_player_df["Bat_f"]))
    print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
    print("Available Players:", sum(sel_player_df["Available"]))
    print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

    print(sel_player_df.sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'))

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Change opt_strat to efp for player efp Optimisation Strategy


# 2. New EFP Approach

# a. Data Extraction 

In [180]:
# New Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Current Round
    current_round = current_rnd

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 10.51, player_df_raw["exp_pts"] + 4.53)
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

    # Aggregate by player name per round
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Round", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    exp_rnd_points=('exp_points',"sum"),
    games_in_round=('Round',"count"))

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# b. Player points and price prediction for each round

In [181]:
# Create Player EFP & Price for each round
if opt_strat == 'efp':
    # Extract Pricing Models
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Create previous game points lag variables for price prediction
    # 1 Game Lag
    player_df_lag_1 = player_df_raw[['Name', 'game_num', 'exp_points']]
    player_df_lag_1['game_num'] = player_df_lag_1['game_num']
    player_df_lag_1 = player_df_lag_1.rename(columns={"exp_points": "exp_points_lag_1"})

    # 2 Game Lag
    player_df_lag_2 = player_df_raw[['Name', 'game_num', 'exp_points']]
    player_df_lag_2['game_num'] = player_df_lag_2['game_num'] + 1
    player_df_lag_2 = player_df_lag_2.rename(columns={"exp_points": "exp_points_lag_2"})

    # 3 Game Lag
    player_df_lag_3 = player_df_raw[['Name', 'game_num', 'exp_points']]
    player_df_lag_3['game_num'] = player_df_lag_3['game_num'] + 2
    player_df_lag_3 = player_df_lag_3.rename(columns={"exp_points": "exp_points_lag_3"})

    # Merge lag variables to main player df
    player_df_lags = player_df_raw.copy()
    player_df_lags = pd.merge(player_df_lags, player_df_lag_1, left_on = ["Name", "game_num"], right_on = ["Name", "game_num"], how = "left")
    player_df_lags = pd.merge(player_df_lags, player_df_lag_2, left_on = ["Name", "game_num"], right_on = ["Name", "game_num"], how = "left")
    player_df_lags = pd.merge(player_df_lags, player_df_lag_3, left_on = ["Name", "game_num"], right_on = ["Name", "game_num"], how = "left")

    # Create moving average variables for price prediction
    player_df_lags['seas_avg_games_pts'] = player_df_lags[['exp_points_lag_1']]
    player_df_lags['last_2_games_ma_pts'] = player_df_lags[['exp_points_lag_1', 'exp_points_lag_2']].mean(axis=1)
    player_df_lags['last_3_games_ma_pts'] = player_df_lags[['exp_points_lag_1', 'exp_points_lag_2', 'exp_points_lag_3']].mean(axis=1)

    # Keep only last game per player per round
    player_df_lags = player_df_lags.sort_values(by=['Name', 'Round', 'game_num'], ascending=[True, True, False])
    player_df_lags = player_df_lags.drop_duplicates(subset=['Name', 'Round'], keep='first')
    player_df_lags = player_df_lags[['Name', 'Price','Round', 'game_num', 'seas_avg_games_pts', 'last_2_games_ma_pts', 'last_3_games_ma_pts']]
    player_df_lags = player_df_lags.rename(columns={"Price": "price_pre"})

    # Split into game count dataframes
    player_df_price_1 = player_df_lags[player_df_lags['game_num'] == 1][['Name', 'price_pre','Round', 'game_num', 'seas_avg_games_pts']]
    player_df_price_2 = player_df_lags[player_df_lags['game_num'] == 2][['Name', 'price_pre','Round', 'game_num', 'last_2_games_ma_pts']]
    player_df_price_3 = player_df_lags[player_df_lags['game_num'] >= 3][['Name', 'price_pre','Round', 'game_num', 'last_3_games_ma_pts']]

    # Predict Prices
    player_df_price_1['Price_Pred'] = price_model_obj_1.predict(player_df_price_1[['price_pre','seas_avg_games_pts']])
    player_df_price_2['Price_Pred'] = price_model_obj_2.predict(player_df_price_2[['price_pre','last_2_games_ma_pts']])
    player_df_price_3['Price_Pred'] = price_model_obj_3.predict(player_df_price_3[['price_pre','last_3_games_ma_pts']])

    player_df_price_1 = player_df_price_1[['Name', 'Round', 'game_num', 'Price_Pred']]
    player_df_price_2 = player_df_price_2[['Name', 'Round', 'game_num', 'Price_Pred']]
    player_df_price_3 = player_df_price_3[['Name', 'Round', 'game_num', 'Price_Pred']]

    # Combine price dataframes
    player_df_price = pd.concat([player_df_price_1, player_df_price_2, player_df_price_3], ignore_index=True)
    
    # Increase round by 1 for price prediction to align with next round
    player_df_price['Round'] = player_df_price['Round'] + 1
    player_df_pred_price = player_df_price[['Name', 'Round', 'Price_Pred']]

    # Merge price predictions back to main player df
    player_df = pd.merge(player_df, player_df_pred_price, left_on = ["Name", "Round"], right_on = ["Name", "Round"], how = "left")
    
    # Player price is actual price for current round, predicted price for subsequent rounds
    player_df['Price'] = np.where(player_df['Round'] == current_rnd, player_df['Price'], player_df['Price_Pred'])
    player_df = player_df.drop(columns=['Price_Pred'])

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")  


C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\299116795.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df_lag_1['game_num'] = player_df_lag_1['game_num']
C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\299116795.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df_lag_2['game_num'] = player_df_lag_2['game_num'] + 1
C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\299116795.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usi

# c. Create separate round df for round based optimisation

In [182]:
# Split player point dataframe by round for optimisation
if opt_strat == 'efp':
    player_df_r1 = player_df[player_df['Round'] == 1].reset_index(drop=True)
    player_df_r2 = player_df[player_df['Round'] == 2].reset_index(drop=True)
    player_df_r3 = player_df[player_df['Round'] == 3].reset_index(drop=True)
    player_df_r4 = player_df[player_df['Round'] == 4].reset_index(drop=True)
    player_df_r5 = player_df[player_df['Round'] == 5].reset_index(drop=True)
    player_df_r6 = player_df[player_df['Round'] == 6].reset_index(drop=True)
    player_df_r7 = player_df[player_df['Round'] == 7].reset_index(drop=True)
    player_df_r8 = player_df[player_df['Round'] == 8].reset_index(drop=True)
    player_df_r9 = player_df[player_df['Round'] == 9].reset_index(drop=True)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")  

# d. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [183]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # a. EFP Optimisation Variables Setup
    # Round 1
    points_r1 = player_df_r1["exp_rnd_points"]
    price_r1 = player_df_r1["Price"]
    weight_r1 = player_df_r1["weight"]
    in_team_r1 = player_df_r1["In_Team"]
    available_r1 = player_df_r1["Available"]
    wk_weight_r1 = player_df_r1["Wk_f"]
    bat_weight_r1 = player_df_r1["Bat_f"]
    bowl_weight_r1 = player_df_r1["Bowl_f"]
    play_cnt_r1, total_player_r1 = 12, range(len(price_r1))
    wk_cnt_r1, total_wk_r1 = 1, range(len(price_r1))
    bat_cnt_r1, total_bat_r1 = 6, range(len(price_r1))
    bowl_cnt_r1, total_bowl_r1 = 5, range(len(price_r1))
    budget_r1, total_budget_r1 = 1783500, range(len(price_r1))

    # Round 2
    points_r2 = player_df_r2["exp_rnd_points"]
    price_r2 = player_df_r2["Price"]
    weight_r2 = player_df_r2["weight"]
    in_team_r2 = player_df_r2["In_Team"]
    available_r2 = player_df_r2["Available"]
    wk_weight_r2 = player_df_r2["Wk_f"]
    bat_weight_r2 = player_df_r2["Bat_f"]
    bowl_weight_r2 = player_df_r2["Bowl_f"]
    play_cnt_r2, total_player_r2 = 12, range(len(price_r2))
    wk_cnt_r2, total_wk_r2 = 1, range(len(price_r2))
    bat_cnt_r2, total_bat_r2 = 6, range(len(price_r2))
    bowl_cnt_r2, total_bowl_r2 = 5, range(len(price_r2))
    budget_r2, total_budget_r2 = 1783500, range(len(price_r2))
    team_play_cnt_r2, total_team_player_r2 = 9, range(len(price_r2)) # At least 9 players from round 1 to be in round 2 team

    # Round 3
    points_r3 = player_df_r3["exp_rnd_points"]
    price_r3 = player_df_r3["Price"]
    weight_r3 = player_df_r3["weight"]
    in_team_r3 = player_df_r3["In_Team"]
    available_r3 = player_df_r3["Available"]
    wk_weight_r3 = player_df_r3["Wk_f"]
    bat_weight_r3 = player_df_r3["Bat_f"]
    bowl_weight_r3 = player_df_r3["Bowl_f"]
    play_cnt_r3, total_player_r3 = 12, range(len(price_r3))
    wk_cnt_r3, total_wk_r3 = 1, range(len(price_r3))
    bat_cnt_r3, total_bat_r3 = 6, range(len(price_r3))
    bowl_cnt_r3, total_bowl_r3 = 5, range(len(price_r3))
    budget_r3, total_budget_r3 = 1783500, range(len(price_r3))
    team_play_cnt_r3, total_team_player_r3 = 9, range(len(price_r3)) # At least 9 players from round 2 to be in round 3 team

    # Round 4
    points_r4 = player_df_r4["exp_rnd_points"]
    price_r4 = player_df_r4["Price"]
    weight_r4 = player_df_r4["weight"]
    in_team_r4 = player_df_r4["In_Team"]
    available_r4 = player_df_r4["Available"]
    wk_weight_r4 = player_df_r4["Wk_f"]
    bat_weight_r4 = player_df_r4["Bat_f"]
    bowl_weight_r4 = player_df_r4["Bowl_f"]
    play_cnt_r4, total_player_r4 = 12, range(len(price_r4))
    wk_cnt_r4, total_wk_r4 = 1, range(len(price_r4))
    bat_cnt_r4, total_bat_r4 = 6, range(len(price_r4))
    bowl_cnt_r4, total_bowl_r4 = 5, range(len(price_r4))
    budget_r4, total_budget_r4 = 1783500, range(len(price_r4))
    team_play_cnt_r4, total_team_player_r4 = 9, range(len(price_r4)) # At least 9 players from round 3 to be in round 4 team

    # Round 5
    points_r5 = player_df_r5["exp_rnd_points"]
    price_r5 = player_df_r5["Price"]
    weight_r5 = player_df_r5["weight"]
    in_team_r5 = player_df_r5["In_Team"]
    available_r5 = player_df_r5["Available"]
    wk_weight_r5 = player_df_r5["Wk_f"]
    bat_weight_r5 = player_df_r5["Bat_f"]
    bowl_weight_r5 = player_df_r5["Bowl_f"]
    play_cnt_r5, total_player_r5 = 12, range(len(price_r5))
    wk_cnt_r5, total_wk_r5 = 1, range(len(price_r5))
    bat_cnt_r5, total_bat_r5 = 6, range(len(price_r5))
    bowl_cnt_r5, total_bowl_r5 = 5, range(len(price_r5))
    budget_r5, total_budget_r5 = 1783500, range(len(price_r5))
    team_play_cnt_r5, total_team_player_r5 = 9, range(len(price_r5)) # At least 9 players from round 4 to be in round 5 team

    # Round 6
    points_r6 = player_df_r6["exp_rnd_points"]
    price_r6 = player_df_r6["Price"]
    weight_r6 = player_df_r6["weight"]
    in_team_r6 = player_df_r6["In_Team"]
    available_r6 = player_df_r6["Available"]
    wk_weight_r6 = player_df_r6["Wk_f"]
    bat_weight_r6 = player_df_r6["Bat_f"]
    bowl_weight_r6 = player_df_r6["Bowl_f"]
    play_cnt_r6, total_player_r6 = 12, range(len(price_r6))
    wk_cnt_r6, total_wk_r6 = 1, range(len(price_r6))
    bat_cnt_r6, total_bat_r6 = 6, range(len(price_r6))
    bowl_cnt_r6, total_bowl_r6 = 5, range(len(price_r6))
    budget_r6, total_budget_r6 = 1783500, range(len(price_r6))
    team_play_cnt_r6, total_team_player_r6 = 9, range(len(price_r6)) # At least 9 players from round 5 to be in round 6 team

    # Round 7
    points_r7 = player_df_r7["exp_rnd_points"]
    price_r7 = player_df_r7["Price"]
    weight_r7 = player_df_r7["weight"]
    in_team_r7 = player_df_r7["In_Team"]
    available_r7 = player_df_r7["Available"]
    wk_weight_r7 = player_df_r7["Wk_f"]
    bat_weight_r7 = player_df_r7["Bat_f"]
    bowl_weight_r7 = player_df_r7["Bowl_f"]
    play_cnt_r7, total_player_r7 = 12, range(len(price_r7))
    wk_cnt_r7, total_wk_r7 = 1, range(len(price_r7))
    bat_cnt_r7, total_bat_r7 = 6, range(len(price_r7))
    bowl_cnt_r7, total_bowl_r7 = 5, range(len(price_r7))
    budget_r7, total_budget_r7 = 1783500, range(len(price_r7))
    team_play_cnt_r7, total_team_player_r7 = 9, range(len(price_r7)) # At least 9 players from round 6 to be in round 7 team

    # Round 8
    points_r8 = player_df_r8["exp_rnd_points"]
    price_r8 = player_df_r8["Price"]
    weight_r8 = player_df_r8["weight"]
    in_team_r8 = player_df_r8["In_Team"]
    available_r8 = player_df_r8["Available"]
    wk_weight_r8 = player_df_r8["Wk_f"]
    bat_weight_r8 = player_df_r8["Bat_f"]
    bowl_weight_r8 = player_df_r8["Bowl_f"]
    play_cnt_r8, total_player_r8 = 12, range(len(price_r8))
    wk_cnt_r8, total_wk_r8 = 1, range(len(price_r8))
    bat_cnt_r8, total_bat_r8 = 6, range(len(price_r8))
    bowl_cnt_r8, total_bowl_r8 = 5, range(len(price_r8))
    budget_r8, total_budget_r8 = 1783500, range(len(price_r8))
    team_play_cnt_r8, total_team_player_r8 = 9, range(len(price_r8)) # At least 9 players from round 7 to be in round 8 team

    # Round 9
    points_r9 = player_df_r9["exp_rnd_points"]
    price_r9 = player_df_r9["Price"]
    weight_r9 = player_df_r9["weight"]
    in_team_r9 = player_df_r9["In_Team"]
    available_r9 = player_df_r9["Available"]
    wk_weight_r9 = player_df_r9["Wk_f"]
    bat_weight_r9 = player_df_r9["Bat_f"]
    bowl_weight_r9 = player_df_r9["Bowl_f"]
    play_cnt_r9, total_player_r9 = 12, range(len(price_r9))
    wk_cnt_r9, total_wk_r9 = 1, range(len(price_r9))
    bat_cnt_r9, total_bat_r9 = 6, range(len(price_r9))
    bowl_cnt_r9, total_bowl_r9 = 5, range(len(price_r9))
    budget_r9, total_budget_r9 = 1783500, range(len(price_r9))
    team_play_cnt_r9, total_team_player_r9 = 9, range(len(price_r9)) # At least 9 players from round 8 to be in round 9 team

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

In [184]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # b. MIP Optimsation Setup
    # a. initialize optimisation parameters
    m = Model("knapsack")
    # x is player selected variable, y is captain selected variable (2x points)
    # Round 1
    x_r1 = [m.add_var(var_type=BINARY) for i in total_player_r1]
    y_r1 = [m.add_var(var_type=BINARY) for i in total_player_r1]
    # Round 2
    x_r2 = [m.add_var(var_type=BINARY) for i in total_player_r2]
    y_r2 = [m.add_var(var_type=BINARY) for i in total_player_r2]
    # Round 3
    x_r3 = [m.add_var(var_type=BINARY) for i in total_player_r3]
    y_r3 = [m.add_var(var_type=BINARY) for i in total_player_r3]
    # Round 4
    x_r4 = [m.add_var(var_type=BINARY) for i in total_player_r4]
    y_r4 = [m.add_var(var_type=BINARY) for i in total_player_r4]
    # Round 5
    x_r5 = [m.add_var(var_type=BINARY) for i in total_player_r5]
    y_r5 = [m.add_var(var_type=BINARY) for i in total_player_r5]
    # Round 6
    x_r6 = [m.add_var(var_type=BINARY) for i in total_player_r6]
    y_r6 = [m.add_var(var_type=BINARY) for i in total_player_r6]
    # Round 7
    x_r7 = [m.add_var(var_type=BINARY) for i in total_player_r7]
    y_r7 = [m.add_var(var_type=BINARY) for i in total_player_r7]
    # Round 8
    x_r8 = [m.add_var(var_type=BINARY) for i in total_player_r8]
    y_r8 = [m.add_var(var_type=BINARY) for i in total_player_r8]
    # Round 9
    x_r9 = [m.add_var(var_type=BINARY) for i in total_player_r9]
    y_r9 = [m.add_var(var_type=BINARY) for i in total_player_r9]

    # linking constraints between consecutive rounds
    # Initialize z_shared for round 1 to round 2 linking
    name_to_idx_r1 = {name: idx for idx, name in enumerate(player_df_r1['Name'].astype(str))}
    z_shared = {}
    for j, name in enumerate(player_df_r2['Name'].astype(str)):
        if name in name_to_idx_r1:
            i = name_to_idx_r1[name]
            z_shared[(i, j)] = m.add_var(var_type=BINARY)
            
    rounds_data = [
        (player_df_r1, player_df_r2, x_r1, x_r2, z_shared, team_play_cnt_r2, "r1_r2"),
        (player_df_r2, player_df_r3, x_r2, x_r3, {}, team_play_cnt_r3, "r2_r3"),
        (player_df_r3, player_df_r4, x_r3, x_r4, {}, team_play_cnt_r4, "r3_r4"),
        (player_df_r4, player_df_r5, x_r4, x_r5, {}, team_play_cnt_r5, "r4_r5"),
        (player_df_r5, player_df_r6, x_r5, x_r6, {}, team_play_cnt_r6, "r5_r6"),
        (player_df_r6, player_df_r7, x_r6, x_r7, {}, team_play_cnt_r7, "r6_r7"),
        (player_df_r7, player_df_r8, x_r7, x_r8, {}, team_play_cnt_r8, "r7_r8"),
        (player_df_r8, player_df_r9, x_r8, x_r9, {}, team_play_cnt_r9, "r8_r9"),
    ]
    
    for curr_df, next_df, curr_x, next_x, z_dict, team_min, label in rounds_data:
        name_to_idx_curr = {name: idx for idx, name in enumerate(curr_df['Name'].astype(str))}
        shared_pairs_rnd = []
        for j, name in enumerate(next_df['Name'].astype(str)):
            if name in name_to_idx_curr:
                i = name_to_idx_curr[name]
                shared_pairs_rnd.append((i, j))
        
        if shared_pairs_rnd:
            z_rnd = {}
            for (i, j) in shared_pairs_rnd:
                z_rnd[(i, j)] = m.add_var(var_type=BINARY)
                m += z_rnd[(i, j)] <= curr_x[i]
                m += z_rnd[(i, j)] <= next_x[j]
                m += z_rnd[(i, j)] >= curr_x[i] + next_x[j] - 1
            m += xsum(z_rnd[(i, j)] for (i, j) in z_rnd.keys()) >= team_min

    # b. define objective function
    obj_r1 = xsum(points_r1[i]*x_r1[i] + points_r1[i]*y_r1[i] for i in total_player_r1)
    obj_r2 = xsum(points_r2[i]*x_r2[i] + points_r2[i]*y_r2[i] for i in total_player_r2)
    obj_r3 = xsum(points_r3[i]*x_r3[i] + points_r3[i]*y_r3[i] for i in total_player_r3)
    obj_r4 = xsum(points_r4[i]*x_r4[i] + points_r4[i]*y_r4[i] for i in total_player_r4)
    obj_r5 = xsum(points_r5[i]*x_r5[i] + points_r5[i]*y_r5[i] for i in total_player_r5)
    obj_r6 = xsum(points_r6[i]*x_r6[i] + points_r6[i]*y_r6[i] for i in total_player_r6)
    obj_r7 = xsum(points_r7[i]*x_r7[i] + points_r7[i]*y_r7[i] for i in total_player_r7)
    obj_r8 = xsum(points_r8[i]*x_r8[i] + points_r8[i]*y_r8[i] for i in total_player_r8)
    obj_r9 = xsum(points_r9[i]*x_r9[i] + points_r9[i]*y_r9[i] for i in total_player_r9)

    m.objective = maximize(obj_r1 + obj_r2 + obj_r3 + obj_r4 + obj_r5 + obj_r6 + obj_r7 + obj_r8 + obj_r9)
    
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

In [185]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':   
    # c. define constraints
        # Player selection constraints
    # Round 1
    m += xsum(weight_r1[i] * x_r1[i] for i in total_player_r1) == play_cnt_r1
    m += xsum(y_r1[i] for i in total_player_r1) == 1  # Only one captain
    # Round 2
    m += xsum(weight_r2[i] * x_r2[i] for i in total_player_r2) == play_cnt_r2
    m += xsum(y_r2[i] for i in total_player_r2) == 1  # Only one captain
    # Round 3
    m += xsum(weight_r3[i] * x_r3[i] for i in total_player_r3) == play_cnt_r3
    m += xsum(y_r3[i] for i in total_player_r3) == 1  # Only one captain
    # Round 4
    m += xsum(weight_r4[i] * x_r4[i] for i in total_player_r4) == play_cnt_r4
    m += xsum(y_r4[i] for i in total_player_r4) == 1  # Only one captain
    # Round 5
    m += xsum(weight_r5[i] * x_r5[i] for i in total_player_r5) == play_cnt_r5
    m += xsum(y_r5[i] for i in total_player_r5) == 1  # Only one captain
    # Round 6
    m += xsum(weight_r6[i] * x_r6[i] for i in total_player_r6) == play_cnt_r6
    m += xsum(y_r6[i] for i in total_player_r6) == 1  # Only one captain
    # Round 7
    m += xsum(weight_r7[i] * x_r7[i] for i in total_player_r7) == play_cnt_r7
    m += xsum(y_r7[i] for i in total_player_r7) == 1  # Only one captain
    # Round 8
    m += xsum(weight_r8[i] * x_r8[i] for i in total_player_r8) == play_cnt_r8
    m += xsum(y_r8[i] for i in total_player_r8) == 1  # Only one captain
    # Round 9
    m += xsum(weight_r9[i] * x_r9[i] for i in total_player_r9) == play_cnt_r9
    m += xsum(y_r9[i] for i in total_player_r9) == 1  # Only one captain

    # captain can only be chosen among selected players
    for i in total_player_r1:
        m += y_r1[i] <= x_r1[i]
    for i in total_player_r2:
        m += y_r2[i] <= x_r2[i]
    for i in total_player_r3:
        m += y_r3[i] <= x_r3[i]
    for i in total_player_r4:
        m += y_r4[i] <= x_r4[i]
    for i in total_player_r5:
        m += y_r5[i] <= x_r5[i]
    for i in total_player_r6:
        m += y_r6[i] <= x_r6[i]
    for i in total_player_r7:
        m += y_r7[i] <= x_r7[i]
    for i in total_player_r8:
        m += y_r8[i] <= x_r8[i]
    for i in total_player_r9:
        m += y_r9[i] <= x_r9[i]

        # Team composition constraints
    # Round 1  
    m += xsum(wk_weight_r1[i] * x_r1[i] for i in total_wk_r1) >= wk_cnt_r1
    m += xsum(bat_weight_r1[i] * x_r1[i] for i in total_bat_r1) >= bat_cnt_r1
    m += xsum(bowl_weight_r1[i] * x_r1[i] for i in total_bowl_r1) >= bowl_cnt_r1
    m += xsum(price_r1[i] * x_r1[i] for i in total_budget_r1) <= budget_r1
    m += xsum(available_r1[i] * x_r1[i] for i in total_player_r1) == play_cnt_r1
    # Round 2  
    m += xsum(wk_weight_r2[i] * x_r2[i] for i in total_wk_r2) >= wk_cnt_r2
    m += xsum(bat_weight_r2[i] * x_r2[i] for i in total_bat_r2) >= bat_cnt_r2
    m += xsum(bowl_weight_r2[i] * x_r2[i] for i in total_bowl_r2) >= bowl_cnt_r2
    m += xsum(price_r2[i] * x_r2[i] for i in total_budget_r2) <= budget_r2
    m += xsum(available_r2[i] * x_r2[i] for i in total_player_r2) == play_cnt_r2
    # Round 3
    m += xsum(wk_weight_r3[i] * x_r3[i] for i in total_wk_r3) >= wk_cnt_r3
    m += xsum(bat_weight_r3[i] * x_r3[i] for i in total_bat_r3) >= bat_cnt_r3
    m += xsum(bowl_weight_r3[i] * x_r3[i] for i in total_bowl_r3) >= bowl_cnt_r3
    m += xsum(price_r3[i] * x_r3[i] for i in total_budget_r3) <= budget_r3
    m += xsum(available_r3[i] * x_r3[i] for i in total_player_r3) == play_cnt_r3
    # Round 4
    m += xsum(wk_weight_r4[i] * x_r4[i] for i in total_wk_r4) >= wk_cnt_r4
    m += xsum(bat_weight_r4[i] * x_r4[i] for i in total_bat_r4) >= bat_cnt_r4
    m += xsum(bowl_weight_r4[i] * x_r4[i] for i in total_bowl_r4) >= bowl_cnt_r4
    m += xsum(price_r4[i] * x_r4[i] for i in total_budget_r4) <= budget_r4
    m += xsum(available_r4[i] * x_r4[i] for i in total_player_r4) == play_cnt_r4
    # Round 5
    m += xsum(wk_weight_r5[i] * x_r5[i] for i in total_wk_r5) >= wk_cnt_r5
    m += xsum(bat_weight_r5[i] * x_r5[i] for i in total_bat_r5) >= bat_cnt_r5
    m += xsum(bowl_weight_r5[i] * x_r5[i] for i in total_bowl_r5) >= bowl_cnt_r5
    m += xsum(price_r5[i] * x_r5[i] for i in total_budget_r5) <= budget_r5
    m += xsum(available_r5[i] * x_r5[i] for i in total_player_r5) == play_cnt_r5
    # Round 6
    m += xsum(wk_weight_r6[i] * x_r6[i] for i in total_wk_r6) >= wk_cnt_r6
    m += xsum(bat_weight_r6[i] * x_r6[i] for i in total_bat_r6) >= bat_cnt_r6
    m += xsum(bowl_weight_r6[i] * x_r6[i] for i in total_bowl_r6) >= bowl_cnt_r6
    m += xsum(price_r6[i] * x_r6[i] for i in total_budget_r6) <= budget_r6
    m += xsum(available_r6[i] * x_r6[i] for i in total_player_r6) == play_cnt_r6
    # Round 7
    m += xsum(wk_weight_r7[i] * x_r7[i] for i in total_wk_r7) >= wk_cnt_r7
    m += xsum(bat_weight_r7[i] * x_r7[i] for i in total_bat_r7) >= bat_cnt_r7
    m += xsum(bowl_weight_r7[i] * x_r7[i] for i in total_bowl_r7) >= bowl_cnt_r7
    m += xsum(price_r7[i] * x_r7[i] for i in total_budget_r7) <= budget_r7
    m += xsum(available_r7[i] * x_r7[i] for i in total_player_r7) == play_cnt_r7
    # Round 8
    m += xsum(wk_weight_r8[i] * x_r8[i] for i in total_wk_r8) >= wk_cnt_r8
    m += xsum(bat_weight_r8[i] * x_r8[i] for i in total_bat_r8) >= bat_cnt_r8
    m += xsum(bowl_weight_r8[i] * x_r8[i] for i in total_bowl_r8) >= bowl_cnt_r8
    m += xsum(price_r8[i] * x_r8[i] for i in total_budget_r8) <= budget_r8
    m += xsum(available_r8[i] * x_r8[i] for i in total_player_r8) == play_cnt_r8
    # Round 9
    m += xsum(wk_weight_r9[i] * x_r9[i] for i in total_wk_r9) >= wk_cnt_r9
    m += xsum(bat_weight_r9[i] * x_r9[i] for i in total_bat_r9) >= bat_cnt_r9
    m += xsum(bowl_weight_r9[i] * x_r9[i] for i in total_bowl_r9) >= bowl_cnt_r9
    m += xsum(price_r9[i] * x_r9[i] for i in total_budget_r9) <= budget_r9
    m += xsum(available_r9[i] * x_r9[i] for i in total_player_r9) == play_cnt_r9

    # d. solve optimisation
    m.optimize()
    # Round 1 
    selected_r1 = [i for i in total_player_r1 if x_r1[i].x >= 0.99]
    captained_r1 = [i for i in total_player_r1 if y_r1[i].x >= 0.99]
    # Round 2
    selected_r2 = [i for i in total_player_r2 if x_r2[i].x >= 0.99]
    captained_r2 = [i for i in total_player_r2 if y_r2[i].x >= 0.99]
    # Round 3
    selected_r3 = [i for i in total_player_r3 if x_r3[i].x >= 0.99]
    captained_r3 = [i for i in total_player_r3 if y_r3[i].x >= 0.99]
    # Round 4
    selected_r4 = [i for i in total_player_r4 if x_r4[i].x >= 0.99]
    captained_r4 = [i for i in total_player_r4 if y_r4[i].x >= 0.99]
    # Round 5
    selected_r5 = [i for i in total_player_r5 if x_r5[i].x >= 0.99]
    captained_r5 = [i for i in total_player_r5 if y_r5[i].x >= 0.99]
    # Round 6
    selected_r6 = [i for i in total_player_r6 if x_r6[i].x >= 0.99]
    captained_r6 = [i for i in total_player_r6 if y_r6[i].x >= 0.99]
    # Round 7
    selected_r7 = [i for i in total_player_r7 if x_r7[i].x >= 0.99]
    captained_r7 = [i for i in total_player_r7 if y_r7[i].x >= 0.99]
    # Round 8
    selected_r8 = [i for i in total_player_r8 if x_r8[i].x >= 0.99]
    captained_r8 = [i for i in total_player_r8 if y_r8[i].x >= 0.99]
    # Round 9
    selected_r9 = [i for i in total_player_r9 if x_r9[i].x >= 0.99]
    captained_r9 = [i for i in total_player_r9 if y_r9[i].x >= 0.99]

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

In [186]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':    
    # e. Optimisation Results
    print("Selected items (rnd 1): {}".format(selected_r1))
    print("Captain selected (rnd 1): {}".format(captained_r1))

    print("Selected items (rnd 2): {}".format(selected_r2))
    print("Captain selected (rnd 2): {}".format(captained_r2))

    # Optimal Team Output
    # Round 1
    sel_player_df_r1 = player_df_r1.iloc[selected_r1]
    sel_captain_df_r1 = player_df_r1.iloc[captained_r1]
    print("Total Expected Points (rnd 1):", sum(sel_player_df_r1["exp_rnd_points"]) + sum(sel_captain_df_r1["exp_rnd_points"]))
    print("Total Team Cost (rnd 1):", sum(sel_player_df_r1["Price"]))
    print("Captain (rnd 1):", sel_captain_df_r1["Name"].values[0])
    print("Current Players Remaining (rnd 1):", sum(sel_player_df_r1["In_Team"]))

    # Round 2
    sel_player_df_r2 = player_df_r2.iloc[selected_r2]
    sel_captain_df_r2 = player_df_r2.iloc[captained_r2]
    # In Team Flag
    sel_names_r1 = player_df_r1.iloc[selected_r1]['Name'].astype(str).tolist() if selected_r1 else []
    sel_player_df_r2['In_Team'] = sel_player_df_r2['Name'].astype(str).isin(sel_names_r1).astype(int)

    print("Total Expected Points (rnd 2):", sum(sel_player_df_r2["exp_rnd_points"]) + sum(sel_captain_df_r2["exp_rnd_points"]))
    print("Total Team Cost (rnd 2):", sum(sel_player_df_r2["Price"]))
    print("Captain (rnd 2):", sel_captain_df_r2["Name"].values[0])
    print("Current Players Remaining (rnd 2):", sum(sel_player_df_r2["In_Team"]))

    # Round 3
    sel_player_df_r3 = player_df_r3.iloc[selected_r3]
    sel_captain_df_r3 = player_df_r3.iloc[captained_r3]
    # In Team Flag
    sel_names_r2 = player_df_r2.iloc[selected_r2]['Name'].astype(str).tolist() if selected_r2 else []
    sel_player_df_r3['In_Team'] = sel_player_df_r3['Name'].astype(str).isin(sel_names_r2).astype(int)
    print("Total Expected Points (rnd 3):", sum(sel_player_df_r3["exp_rnd_points"]) + sum(sel_captain_df_r3["exp_rnd_points"]))
    print("Total Team Cost (rnd 3):", sum(sel_player_df_r3["Price"]))
    print("Captain (rnd 3):", sel_captain_df_r3["Name"].values[0])
    print("Current Players Remaining (rnd 3):", sum(sel_player_df_r3["In_Team"]))
    # Round 4
    sel_player_df_r4 = player_df_r4.iloc[selected_r4]
    sel_captain_df_r4 = player_df_r4.iloc[captained_r4]
    # In Team Flag
    sel_names_r3 = player_df_r3.iloc[selected_r3]['Name'].astype(str).tolist() if selected_r3 else []
    sel_player_df_r4['In_Team'] = sel_player_df_r4['Name'].astype(str).isin(sel_names_r3).astype(int)
    print("Total Expected Points (rnd 4):", sum(sel_player_df_r4["exp_rnd_points"]) + sum(sel_captain_df_r4["exp_rnd_points"]))
    print("Total Team Cost (rnd 4):", sum(sel_player_df_r4["Price"]))
    print("Captain (rnd 4):", sel_captain_df_r4["Name"].values[0])
    print("Current Players Remaining (rnd 4):", sum(sel_player_df_r4["In_Team"]))
    # Round 5
    sel_player_df_r5 = player_df_r5.iloc[selected_r5]
    sel_captain_df_r5 = player_df_r5.iloc[captained_r5]
    # In Team Flag  
    sel_names_r4 = player_df_r4.iloc[selected_r4]['Name'].astype(str).tolist() if selected_r4 else []
    sel_player_df_r5['In_Team'] = sel_player_df_r5['Name'].astype(str).isin(sel_names_r4).astype(int)
    print("Total Expected Points (rnd 5):", sum(sel_player_df_r5["exp_rnd_points"]) + sum(sel_captain_df_r5["exp_rnd_points"]))
    print("Total Team Cost (rnd 5):", sum(sel_player_df_r5["Price"]))
    print("Captain (rnd 5):", sel_captain_df_r5["Name"].values[0])
    print("Current Players Remaining (rnd 5):", sum(sel_player_df_r5["In_Team"]))
    # Round 6
    sel_player_df_r6 = player_df_r6.iloc[selected_r6]
    sel_captain_df_r6 = player_df_r6.iloc[captained_r6]
    # In Team Flag
    sel_names_r5 = player_df_r5.iloc[selected_r5]['Name'].astype(str).tolist() if selected_r5 else []
    sel_player_df_r6['In_Team'] = sel_player_df_r6['Name'].astype(str).isin(sel_names_r5).astype(int)
    print("Total Expected Points (rnd 6):", sum(sel_player_df_r6["exp_rnd_points"]) + sum(sel_captain_df_r6["exp_rnd_points"]))
    print("Total Team Cost (rnd 6):", sum(sel_player_df_r6["Price"]))
    print("Captain (rnd 6):", sel_captain_df_r6["Name"].values[0])
    print("Current Players Remaining (rnd 6):", sum(sel_player_df_r6["In_Team"]))
    # Round 7
    sel_player_df_r7 = player_df_r7.iloc[selected_r7]
    sel_captain_df_r7 = player_df_r7.iloc[captained_r7]
    # In Team Flag
    sel_names_r6 = player_df_r6.iloc[selected_r6]['Name'].astype(str).tolist() if selected_r6 else []
    sel_player_df_r7['In_Team'] = sel_player_df_r7['Name'].astype(str).isin(sel_names_r6).astype(int)
    print("Total Expected Points (rnd 7):", sum(sel_player_df_r7["exp_rnd_points"]) + sum(sel_captain_df_r7["exp_rnd_points"]))
    print("Total Team Cost (rnd 7):", sum(sel_player_df_r7["Price"]))
    print("Captain (rnd 7):", sel_captain_df_r7["Name"].values[0])
    print("Current Players Remaining (rnd 7):", sum(sel_player_df_r7["In_Team"]))
    # Round 8
    sel_player_df_r8 = player_df_r8.iloc[selected_r8]
    sel_captain_df_r8 = player_df_r8.iloc[captained_r8]
    # In Team Flag
    sel_names_r7 = player_df_r7.iloc[selected_r7]['Name'].astype(str).tolist() if selected_r7 else []
    sel_player_df_r8['In_Team'] = sel_player_df_r8['Name'].astype(str).isin(sel_names_r7).astype(int)
    print("Total Expected Points (rnd 8):", sum(sel_player_df_r8["exp_rnd_points"]) + sum(sel_captain_df_r8["exp_rnd_points"]))
    print("Total Team Cost (rnd 8):", sum(sel_player_df_r8["Price"]))
    print("Captain (rnd 8):", sel_captain_df_r8["Name"].values[0])
    print("Current Players Remaining (rnd 8):", sum(sel_player_df_r8["In_Team"]))
    # Round 9  
    sel_player_df_r9 = player_df_r9.iloc[selected_r9]
    sel_captain_df_r9 = player_df_r9.iloc[captained_r9]
    # In Team Flag
    sel_names_r8 = player_df_r7.iloc[selected_r8]['Name'].astype(str).tolist() if selected_r8 else []
    sel_player_df_r9['In_Team'] = sel_player_df_r9['Name'].astype(str).isin(sel_names_r8).astype(int)
    print("Total Expected Points (rnd 8):", sum(sel_player_df_r9["exp_rnd_points"]) + sum(sel_captain_df_r9["exp_rnd_points"]))
    print("Total Team Cost (rnd 8):", sum(sel_player_df_r9["Price"]))
    print("Captain (rnd 8):", sel_captain_df_r9["Name"].values[0])
    print("Current Players Remaining (rnd 8):", sum(sel_player_df_r9["In_Team"]))

    # Combine Selected Player DataFrames
    sel_player_df = pd.concat([sel_player_df_r1, sel_player_df_r2, sel_player_df_r3, sel_player_df_r4,
                               sel_player_df_r5, sel_player_df_r6, sel_player_df_r7,sel_player_df_r8, sel_player_df_r9], ignore_index=True)

    print("Total Expected Points:", sum(sel_player_df["exp_rnd_points"]) + 
                                    sum(sel_captain_df_r1["exp_rnd_points"]) + 
                                    sum(sel_captain_df_r2["exp_rnd_points"]) +
                                    sum(sel_captain_df_r3["exp_rnd_points"]) +
                                    sum(sel_captain_df_r4["exp_rnd_points"]) +
                                    sum(sel_captain_df_r5["exp_rnd_points"]) +
                                    sum(sel_captain_df_r6["exp_rnd_points"]) +
                                    sum(sel_captain_df_r7["exp_rnd_points"]) +
                                    sum(sel_captain_df_r8["exp_rnd_points"]) +
                                    sum(sel_captain_df_r9["exp_rnd_points"]))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_new_optimal_pre_tourny_team.csv'), index=False)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Selected items (rnd 1): [10, 25, 30, 48, 53, 87, 88, 91, 97, 109, 118, 137]
Captain selected (rnd 1): [10]
Selected items (rnd 2): [25, 30, 53, 87, 88, 90, 91, 97, 99, 109, 121, 137]
Captain selected (rnd 2): [121]
Total Expected Points (rnd 1): 1151.027252
Total Team Cost (rnd 1): 1764600.0
Captain (rnd 1): Ben Dwarshuis
Current Players Remaining (rnd 1): 0
Total Expected Points (rnd 2): 951.776831
Total Team Cost (rnd 2): 1719168.3713740108
Captain (rnd 2): Spencer Johnson
Current Players Remaining (rnd 2): 9
Total Expected Points (rnd 3): 1032.563066
Total Team Cost (rnd 3): 1740143.5090626262
Captain (rnd 3): Mitch Owen
Current Players Remaining (rnd 3): 9
Total Expected Points (rnd 4): 941.706475
Total Team Cost (rnd 4): 1756730.3478304422
Captain (rnd 4): Cooper Connolly
Current Players Remaining (rnd 4): 9
Total Expected Points (rnd 5): 850.523218
Total Team Cost (rnd 5): nan
Captain (rnd 5): Tom Curran
Current Players Remaining (rnd 5): 9
Total Expected Points (rnd 6): 924.5282

C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\3373952338.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel_player_df_r2['In_Team'] = sel_player_df_r2['Name'].astype(str).isin(sel_names_r1).astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\3373952338.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sel_player_df_r3['In_Team'] = sel_player_df_r3['Name'].astype(str).isin(sel_names_r2).astype(int)
C:\Users\dilan\AppData\Local\Temp\ipykernel_37868\3373952338.py:46: SettingWithCopyWarni

# 3. New Distribution Approach

# a. Data Extraction 

In [187]:
if opt_strat == 'range':
    # Simulated Player Points Data Constructions
    # Extract Player Expected Points and Standard Deviation dataframe
    model_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)
    sim_pts_df = model_pts_df.copy()

    # Generate Simulated Points based on Normal Distribution
    # Function to calculate z score bounds given confidence interval
    def z_score_bounds(confidence_level):
        """
        Returns the lower and upper z-scores for a given confidence level.
        For example, confidence_level = 0.90 for 90%.
        """
        alpha = 1 - confidence_level
        lower = norm.ppf(alpha / 2)
        upper = norm.ppf(1 - alpha / 2)
        return lower, upper

    # Set confidence interval and calculate z score bounds
    conf_int = 0.9
    lower_z_thresh, upper_z_thresh = z_score_bounds(conf_int)
    print(f"For {conf_int*100:.0f}% simulated points confidence interval the lower z score is {lower_z_thresh:.3f} and upper z score is {upper_z_thresh:.3f}")

    # Assign random z score for each df row within bounds and calculate simulated points
    sim_pts_df["z_score"] = np.random.uniform(lower_z_thresh, upper_z_thresh, size=len(sim_pts_df))
    sim_pts_df["sim_points"] = sim_pts_df["mean"] + (sim_pts_df["z_score"] * sim_pts_df["std_dev"])
    sim_pts_df["sim_points"] = sim_pts_df["sim_points"].clip(lower=0).round(0)  # Ensure no negative points

else:
    print("Change opt_strat to range for player fp distribution Optimisation Strategy")

Change opt_strat to range for player fp distribution Optimisation Strategy


In [188]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'range':
    # Data Extraction 
    # Current Round & Optimal Number of Rounds
    current_round = current_rnd
    opt_round = 2

    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture.csv'), low_memory=False)
    # team_fix_df = team_fix_df[team_fix_df.Round >= current_round].dropna()
    # team_fix_df = team_fix_df[team_fix_df.Round <= current_round + opt_round - 1].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    # sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False).drop(["Unnamed: 0", "player"], axis = 1)
    sim_pts_df = pd.read_csv(os.path.join(mock_data_dir,'simulated_mock_data.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, sim_pts_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], how = "left")
    player_df_raw["sim_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["sim_points"] + 10.51, player_df_raw["sim_points"] + 4.53)
    player_df_raw["weight"] = 1

    # Aggregate by player name (calculate total expected points for each player for x number of rounds)
    player_df = player_df_raw.groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    sim_points=('sim_points',"sum"))

    # Create dataframe for next round expected points (For captain selection)
    player_df_next_rnd = player_df_raw[player_df_raw.Round == current_round].groupby(['Name', 'Price', "Team", "Wk_f", "Bat_f", "Bowl_f", "Role", "weight","Available", "In_Team"], as_index=False).agg(
    next_rnd_sim_points=('sim_points',"sum"))
    player_df_next_rnd = player_df_next_rnd[["Name", "Team", "next_rnd_sim_points"]]

else:
    print("Change opt_strat to range for player fp distribution Optimisation Strategy")


Change opt_strat to range for player fp distribution Optimisation Strategy


# 2. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,783,500 (As bench costs $216,500)

In [189]:
# Player Distribution Optimisation Variable Setup
if opt_strat == 'range':
    points = player_df["sim_points"]
    price = player_df["Price"]

    weight = player_df["weight"]
    available = player_df["Available"]
    in_team = player_df["In_Team"]
    wk_weight = player_df["Wk_f"]
    bat_weight = player_df["Bat_f"]
    bowl_weight = player_df["Bowl_f"]

    play_cnt, total_player = 12, range(len(price))
    # team_play_cnt, total_team_player = 0, range(len(price)) # IGNORE as pre tourny no players in team
    wk_cnt, total_wk = 1, range(len(price))
    bat_cnt, total_bat = 6, range(len(price))
    bowl_cnt, total_bowl = 5, range(len(price))
    budget, total_budget = 1783500, range(len(price))

    # MIP Optimsation Setup
    m = Model("knapsack")
    x = [m.add_var(var_type=BINARY) for i in total_player]
    print(x)

    m.objective = maximize(xsum(points[i]*x[i] for i in total_player))

    m += xsum(weight[i] * x[i] for i in total_player) == play_cnt
    m += xsum(wk_weight[i] * x[i] for i in total_wk) >= wk_cnt
    m += xsum(bat_weight[i] * x[i] for i in total_bat) >= bat_cnt
    m += xsum(bowl_weight[i] * x[i] for i in total_bowl) >= bowl_cnt
    m += xsum(price[i] * x[i] for i in total_budget) <= budget
    m += xsum(available[i] * x[i] for i in total_player) == play_cnt
    # m += xsum(in_team[i] * x[i] for i in total_team_player) >= team_play_cnt

    # Optimisation Process & Results
    m.optimize() 
    selected = [i for i in total_player if x[i].x >= 0.99]
    print("selected items: {}".format(selected))

    # Optimal Team Output
    sel_player_df = player_df.iloc[selected]
    sel_player_df = pd.merge(sel_player_df, player_df_next_rnd, left_on = ["Name","Team"], right_on = ["Name","Team"], how = "left")
    print("Total Simulated Points:", sum(sel_player_df["sim_points"]))
    print("Total Next Round Points:", sum(sel_player_df["next_rnd_sim_points"]))
    print("Total Team Cost:", sum(sel_player_df["Price"]))
    print("Number of Wk:", sum(sel_player_df["Wk_f"]))
    print("Number of Bat:", sum(sel_player_df["Bat_f"]))
    print("Number of Bowl:", sum(sel_player_df["Bowl_f"]))
    print("Available Players:", sum(sel_player_df["Available"]))
    print("Current Players Remaining:", sum(sel_player_df["In_Team"]))

    print(sel_player_df.sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 1].sort_values(by = "Price", ascending = False))
    print(sel_player_df[sel_player_df['In_Team'] == 0].sort_values(by = "Price", ascending = False))

    # Save Optimal Team to CSV
    sel_player_df.to_csv(os.path.join(add_data_directory,'EFP_optimal_pre_tourny_team.csv'))

else:
    print("Change opt_strat to range for player distribution Optimisation Strategy")

Change opt_strat to range for player distribution Optimisation Strategy
